In [1]:
import ctypes
ctypes.CDLL('/usr/lib/aarch64-linux-gnu/libgomp.so.1', mode=ctypes.RTLD_GLOBAL)

import os
import sys
sys.path.append('/usr/local/lib')

import pyrealsense2 as rs
import numpy as np
import cv2
import time
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output, Image as IPythonImage
print("Wszystkie biblioteki załadowane pomyślnie!")

Wszystkie biblioteki załadowane pomyślnie!


DZIALAJACA WERSJA KODU (BEZ PODGLADU PRAWEGO OBIEKTYWU)
---

In [4]:
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.fisheye, 1, 848, 800, rs.format.y8, 30)
config.enable_stream(rs.stream.fisheye, 2, 848, 800, rs.format.y8, 30)
pipe_profile = pipeline.start(config)

fs1 = pipe_profile.get_stream(rs.stream.fisheye, 1).as_video_stream_profile().get_intrinsics()
fs2 = pipe_profile.get_stream(rs.stream.fisheye, 2).as_video_stream_profile().get_intrinsics()
extrinsics = pipe_profile.get_stream(rs.stream.fisheye, 1).get_extrinsics_to(pipe_profile.get_stream(rs.stream.fisheye, 2))

K1 = np.array([[fs1.fx, 0, fs1.ppx], [0, fs1.fy, fs1.ppy], [0, 0, 1]], dtype=np.float64)
D1 = np.array(fs1.coeffs[:4], dtype=np.float64)
K2 = np.array([[fs2.fx, 0, fs2.ppx], [0, fs2.fy, fs2.ppy], [0, 0, 1]], dtype=np.float64)
D2 = np.array(fs2.coeffs[:4], dtype=np.float64)

R = np.array(extrinsics.rotation, dtype=np.float64).reshape(3,3)
T = np.array(extrinsics.translation, dtype=np.float64).reshape(3,1)

P1 = np.dot(K1, np.hstack((np.eye(3), np.zeros((3,1)))))
P2 = np.dot(K2, np.hstack((R, T)))

cx, cy = 424.0, 400.0
p1_raw = np.array([[[cx, cy]]], dtype=np.float32)

lk_params = dict(winSize=(15, 15), maxLevel=2,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, 0.03))

image_widget = widgets.Image(format='jpeg', width=224, height=224)
text_widget = widgets.HTML(value="<b>ODLEGŁOŚĆ: --</b>")

display(image_widget)
display(text_widget)

try:
    while True:
        frames = pipeline.wait_for_frames()
        f1 = frames.get_fisheye_frame(1)
        f2 = frames.get_fisheye_frame(2)
        if not f1 or not f2:
            continue

        img1 = np.ascontiguousarray(np.asanyarray(f1.get_data()).copy())
        img2 = np.ascontiguousarray(np.asanyarray(f2.get_data()).copy())

        p1_undistorted = cv2.fisheye.undistortPoints(p1_raw, K1, D1, P=K1)
        
        p2_raw, st, err = cv2.calcOpticalFlowPyrLK(img1, img2, p1_raw, None, **lk_params)

        distance_output = "<b>ODLEGŁOŚĆ: --</b>"
        
        if st is not None and st[0][0] == 1 and err is not None:
            if err[0][0] > 30.0:
                pass 
            else:
                p2_undistorted = cv2.fisheye.undistortPoints(p2_raw, K2, D2, P=K2)
                point_4d = cv2.triangulatePoints(P1, P2, p1_undistorted, p2_undistorted)
                
                w = point_4d[3][0]
                if abs(w) > 1e-5:
                    point_3d = point_4d[:3] / w
                    distance_meters = float(point_3d[2][0])
                    
                    if 0.1 < distance_meters < 6.0:
                        distance_output = f"<b>ODLEGŁOŚĆ: {distance_meters:.2f} m</b>"

        color_img = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)
        cv2.drawMarker(color_img, (424, 400), (0, 0, 255), markerType=cv2.MARKER_CROSS, markerSize=40, thickness=3)
        
        ultra_small = cv2.resize(color_img, (224, 224), interpolation=cv2.INTER_AREA)
        _, encoded_img = cv2.imencode('.jpg', ultra_small, [int(cv2.IMWRITE_JPEG_QUALITY), 75])
        
        image_widget.value = encoded_img.tobytes()
        text_widget.value = distance_output

        time.sleep(0.05)

except KeyboardInterrupt:
    pass
finally:
    pipeline.stop()
    print("\nKamera zamknięta bezpiecznie.")

Image(value=b'', format='jpeg', height='224', width='224')

HTML(value='<b>ODLEGŁOŚĆ: --</b>')


Kamera zamknięta bezpiecznie.


WERSJA Z POGLADEM PRAWEGO OKA + TUNING PARAMETROW!!!
---


In [2]:
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.fisheye, 1, 848, 800, rs.format.y8, 30)
config.enable_stream(rs.stream.fisheye, 2, 848, 800, rs.format.y8, 30)
pipe_profile = pipeline.start(config)

fs1 = pipe_profile.get_stream(rs.stream.fisheye, 1).as_video_stream_profile().get_intrinsics()
fs2 = pipe_profile.get_stream(rs.stream.fisheye, 2).as_video_stream_profile().get_intrinsics()
extrinsics = pipe_profile.get_stream(rs.stream.fisheye, 1).get_extrinsics_to(pipe_profile.get_stream(rs.stream.fisheye, 2))

K1 = np.array([[fs1.fx, 0, fs1.ppx], [0, fs1.fy, fs1.ppy], [0, 0, 1]], dtype=np.float64)
D1 = np.array(fs1.coeffs[:4], dtype=np.float64)
K2 = np.array([[fs2.fx, 0, fs2.ppx], [0, fs2.fy, fs2.ppy], [0, 0, 1]], dtype=np.float64)
D2 = np.array(fs2.coeffs[:4], dtype=np.float64)

R = np.array(extrinsics.rotation, dtype=np.float64).reshape(3,3)
T = np.array(extrinsics.translation, dtype=np.float64).reshape(3,1)

P1 = np.dot(K1, np.hstack((np.eye(3), np.zeros((3,1)))))
P2 = np.dot(K2, np.hstack((R, T)))
#mniejsza wartosc 'cx' to w lewo , wieksza warotsc 'cy' to w dol
cx, cy = 470.0, 450.0
p1_raw = np.array([[[cx, cy]]], dtype=np.float32)

lk_params = dict(winSize=(31, 31), maxLevel=4,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))

image_widget = widgets.Image(format='jpeg', width=448, height=224)
text_widget = widgets.HTML(value="<b>ODLEGŁOŚĆ: --</b>")

display(image_widget)
display(text_widget)

try:
    while True:
        frames = pipeline.wait_for_frames()
        f1 = frames.get_fisheye_frame(1)
        f2 = frames.get_fisheye_frame(2)
        if not f1 or not f2:
            continue

        img1 = np.ascontiguousarray(np.asanyarray(f1.get_data()).copy())
        img2 = np.ascontiguousarray(np.asanyarray(f2.get_data()).copy())

        p1_undistorted = cv2.fisheye.undistortPoints(p1_raw, K1, D1, P=K1)
        
        p2_raw, st, err = cv2.calcOpticalFlowPyrLK(img1, img2, p1_raw, None, **lk_params)

        distance_output = "<b>ODLEGŁOŚĆ: --</b>"
        
        color_img1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)
        color_img2 = cv2.cvtColor(img2, cv2.COLOR_GRAY2BGR)
        
        cv2.drawMarker(color_img1, (int(cx), int(cy)), (0, 0, 255), markerType=cv2.MARKER_CROSS, markerSize=40, thickness=3)
        
        if st is not None and st[0][0] == 1 and err is not None:
            if err[0][0] > 18.0:
                pass 
            else:
                p2_undistorted = cv2.fisheye.undistortPoints(p2_raw, K2, D2, P=K2)
                point_4d = cv2.triangulatePoints(P1, P2, p1_undistorted, p2_undistorted)
                
                w = point_4d[3][0]
                if abs(w) > 1e-5:
                    point_3d = point_4d[:3] / w
                    distance_meters = float(point_3d[2][0])
                    
                    if 0.1 < distance_meters < 6.0:
                        distance_output = f"<b>ODLEGŁOŚĆ: {distance_meters:.2f} m</b>"

                px2, py2 = int(p2_raw[0][0][0]), int(p2_raw[0][0][1])
                cv2.drawMarker(color_img2, (px2, py2), (255, 0, 0), markerType=cv2.MARKER_CROSS, markerSize=40, thickness=3)

        side_by_side = np.hstack((color_img1, color_img2))
        ultra_small = cv2.resize(side_by_side, (448, 224), interpolation=cv2.INTER_AREA)
        _, encoded_img = cv2.imencode('.jpg', ultra_small, [int(cv2.IMWRITE_JPEG_QUALITY), 75])
        
        image_widget.value = encoded_img.tobytes()
        text_widget.value = distance_output

        time.sleep(0.05)

except KeyboardInterrupt:
    pass
finally:
    pipeline.stop()
    print("\nKamera zamknięta bezpiecznie.")

Image(value=b'', format='jpeg', height='224', width='448')

HTML(value='<b>ODLEGŁOŚĆ: --</b>')


Kamera zamknięta bezpiecznie.


Ulepszony Kod (Z Subpikselami, Pamięcią i Filtrem)
---

In [2]:
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.fisheye, 1, 848, 800, rs.format.y8, 30)
config.enable_stream(rs.stream.fisheye, 2, 848, 800, rs.format.y8, 30)
pipe_profile = pipeline.start(config)

fs1 = pipe_profile.get_stream(rs.stream.fisheye, 1).as_video_stream_profile().get_intrinsics()
fs2 = pipe_profile.get_stream(rs.stream.fisheye, 2).as_video_stream_profile().get_intrinsics()
extrinsics = pipe_profile.get_stream(rs.stream.fisheye, 1).get_extrinsics_to(pipe_profile.get_stream(rs.stream.fisheye, 2))

K1 = np.array([[fs1.fx, 0, fs1.ppx], [0, fs1.fy, fs1.ppy], [0, 0, 1]], dtype=np.float64)
D1 = np.array(fs1.coeffs[:4], dtype=np.float64)
K2 = np.array([[fs2.fx, 0, fs2.ppx], [0, fs2.fy, fs2.ppy], [0, 0, 1]], dtype=np.float64)
D2 = np.array(fs2.coeffs[:4], dtype=np.float64)

R = np.array(extrinsics.rotation, dtype=np.float64).reshape(3,3)
T = np.array(extrinsics.translation, dtype=np.float64).reshape(3,1)

P1 = np.dot(K1, np.hstack((np.eye(3), np.zeros((3,1)))))
P2 = np.dot(K2, np.hstack((R, T)))

cx, cy = 424.0, 400.0
p1_raw = np.array([[[cx, cy]]], dtype=np.float32)


lk_params = dict(winSize=(41, 41), maxLevel=4,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 40, 0.005))


subpix_criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 0.001)


p2_prev = None          
smoothed_dist = None   
alpha = 0.25           

image_widget = widgets.Image(format='jpeg', width=448, height=224)
text_widget = widgets.HTML(value="<b>ODLEGŁOŚĆ: --</b>")

display(image_widget)
display(text_widget)

try:
    while True:
        frames = pipeline.wait_for_frames()
        f1 = frames.get_fisheye_frame(1)
        f2 = frames.get_fisheye_frame(2)
        if not f1 or not f2:
            continue

        img1 = np.ascontiguousarray(np.asanyarray(f1.get_data()).copy())
        img2 = np.ascontiguousarray(np.asanyarray(f2.get_data()).copy())

        p1_undistorted = cv2.fisheye.undistortPoints(p1_raw, K1, D1, P=K1)
        
        initial_guess = p2_prev if p2_prev is not None else p1_raw.copy()
        
        p2_raw, st, err = cv2.calcOpticalFlowPyrLK(
            img1, img2, p1_raw, initial_guess, 
            flags=cv2.OPTFLOW_USE_INITIAL_FLOW if p2_prev is not None else 0, 
            **lk_params
        )

        distance_output = "<b>ODLEGŁOŚĆ: --</b>"
        tracking_success = False
        
        color_img1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)
        color_img2 = cv2.cvtColor(img2, cv2.COLOR_GRAY2BGR)
        
        cv2.drawMarker(color_img1, (int(cx), int(cy)), (0, 0, 255), markerType=cv2.MARKER_CROSS, markerSize=40, thickness=3)
        
        if st is not None and st[0][0] == 1 and err is not None and err[0][0] < 22.0:
            
            # SUBPIKSELE: Wyostrzamy znaleziony punkt w prawym oku
            p2_raw = cv2.cornerSubPix(img2, p2_raw, (11, 11), (-1, -1), subpix_criteria)
            
            p2_undistorted = cv2.fisheye.undistortPoints(p2_raw, K2, D2, P=K2)
            point_4d = cv2.triangulatePoints(P1, P2, p1_undistorted, p2_undistorted)
            
            w = point_4d[3][0]
            if abs(w) > 1e-5:
                point_3d = point_4d[:3] / w
                distance_meters = float(point_3d[2][0])
                
                if 0.1 < distance_meters < 6.0:
      
                    if smoothed_dist is None:
                        smoothed_dist = distance_meters
                    else:
                        smoothed_dist = alpha * distance_meters + (1 - alpha) * smoothed_dist
                        
                    distance_output = f"<b>ODLEGŁOŚĆ: {smoothed_dist:.2f} m</b>"
                    
 
                    p2_prev = p2_raw.copy()
                    tracking_success = True

                    px2, py2 = int(p2_raw[0][0][0]), int(p2_raw[0][0][1])
                    cv2.drawMarker(color_img2, (px2, py2), (255, 0, 0), markerType=cv2.MARKER_CROSS, markerSize=40, thickness=3)

        if not tracking_success:
            p2_prev = None

        side_by_side = np.hstack((color_img1, color_img2))
        ultra_small = cv2.resize(side_by_side, (448, 224), interpolation=cv2.INTER_AREA)
        _, encoded_img = cv2.imencode('.jpg', ultra_small, [int(cv2.IMWRITE_JPEG_QUALITY), 75])
        
        image_widget.value = encoded_img.tobytes()
        text_widget.value = distance_output

        time.sleep(0.04)

except KeyboardInterrupt:
    pass
finally:
    pipeline.stop()
    print("\nKamera zamknięta bezpiecznie.")

Image(value=b'', format='jpeg', height='224', width='448')

HTML(value='<b>ODLEGŁOŚĆ: --</b>')


Kamera zamknięta bezpiecznie.


testing
---

In [2]:
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.fisheye, 1, 848, 800, rs.format.y8, 30)
config.enable_stream(rs.stream.fisheye, 2, 848, 800, rs.format.y8, 30)
pipe_profile = pipeline.start(config)

fs1 = pipe_profile.get_stream(rs.stream.fisheye, 1).as_video_stream_profile().get_intrinsics()
fs2 = pipe_profile.get_stream(rs.stream.fisheye, 2).as_video_stream_profile().get_intrinsics()
extrinsics = pipe_profile.get_stream(rs.stream.fisheye, 1).get_extrinsics_to(pipe_profile.get_stream(rs.stream.fisheye, 2))

K1 = np.array([[fs1.fx, 0, fs1.ppx], [0, fs1.fy, fs1.ppy], [0, 0, 1]], dtype=np.float64)
D1 = np.array(fs1.coeffs[:4], dtype=np.float64)
K2 = np.array([[fs2.fx, 0, fs2.ppx], [0, fs2.fy, fs2.ppy], [0, 0, 1]], dtype=np.float64)
D2 = np.array(fs2.coeffs[:4], dtype=np.float64)

R = np.array(extrinsics.rotation, dtype=np.float64).reshape(3,3)
T = np.array(extrinsics.translation, dtype=np.float64).reshape(3,1)

P1 = np.dot(K1, np.hstack((np.eye(3), np.zeros((3,1)))))
P2 = np.dot(K2, np.hstack((R, T)))

cx, cy = 424.0, 400.0
p1_raw = np.array([[[cx, cy]]], dtype=np.float32)


TW, TH = 40, 40 

SEARCH_PADDING_X = 200
SEARCH_PADDING_Y = 20  

image_widget = widgets.Image(format='jpeg', width=448, height=224)
text_widget = widgets.HTML(value="<b>ODLEGŁOŚĆ: --</b>")

display(image_widget)
display(text_widget)

try:
    while True:
        frames = pipeline.wait_for_frames()
        f1 = frames.get_fisheye_frame(1)
        f2 = frames.get_fisheye_frame(2)
        if not f1 or not f2:
            continue

        img1 = np.ascontiguousarray(np.asanyarray(f1.get_data()).copy())
        img2 = np.ascontiguousarray(np.asanyarray(f2.get_data()).copy())

        x_start_t = int(cx - TW//2)
        y_start_t = int(cy - TH//2)
        template = img1[y_start_t:y_start_t+TH, x_start_t:x_start_t+TW]

        x_min = max(0, int(cx - SEARCH_PADDING_X))
        x_max = min(img2.shape[1], int(cx + SEARCH_PADDING_X))
        y_min = max(0, int(cy - SEARCH_PADDING_Y))
        y_max = min(img2.shape[0], int(cy + SEARCH_PADDING_Y))
        
        search_area = img2[y_min:y_max, x_min:x_max]

        distance_output = "<b>ODLEGŁOŚĆ: --</b>"
        color_img1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)
        color_img2 = cv2.cvtColor(img2, cv2.COLOR_GRAY2BGR)
        
        cv2.drawMarker(color_img1, (int(cx), int(cy)), (0, 0, 255), markerType=cv2.MARKER_CROSS, markerSize=40, thickness=3)

    
        if search_area.shape[0] >= template.shape[0] and search_area.shape[1] >= template.shape[1]:
      
            res = cv2.matchTemplate(search_area, template, cv2.TM_CCOEFF_NORMED)
            min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res)

            if max_val > 0.70:
           
                px2 = x_min + max_loc[0] + TW//2
                py2 = y_min + max_loc[1] + TH//2
                
                p1_undistorted = cv2.fisheye.undistortPoints(p1_raw, K1, D1, P=K1)
                p2_raw = np.array([[[float(px2), float(py2)]]], dtype=np.float32)
                p2_undistorted = cv2.fisheye.undistortPoints(p2_raw, K2, D2, P=K2)
                
                point_4d = cv2.triangulatePoints(P1, P2, p1_undistorted, p2_undistorted)
                
                w = point_4d[3][0]
                if abs(w) > 1e-5:
                    point_3d = point_4d[:3] / w
                    distance_meters = float(point_3d[2][0])
                    
                    if 0.1 < distance_meters < 6.0:
                        distance_output = f"<b>ODLEGŁOŚĆ: {distance_meters:.2f} m</b>"

                cv2.drawMarker(color_img2, (px2, py2), (255, 0, 0), markerType=cv2.MARKER_CROSS, markerSize=40, thickness=3)

        side_by_side = np.hstack((color_img1, color_img2))
        ultra_small = cv2.resize(side_by_side, (448, 224), interpolation=cv2.INTER_AREA)
        _, encoded_img = cv2.imencode('.jpg', ultra_small, [int(cv2.IMWRITE_JPEG_QUALITY), 75])
        
        image_widget.value = encoded_img.tobytes()
        text_widget.value = distance_output

        time.sleep(0.05)

except KeyboardInterrupt:
    pass
finally:
    pipeline.stop()
    print("\nKamera zamknięta bezpiecznie.")

Image(value=b'', format='jpeg', height='224', width='448')

HTML(value='<b>ODLEGŁOŚĆ: --</b>')


Kamera zamknięta bezpiecznie.
